# V-ChartText – Chạy TOÀN BỘ baseline Mức 1 trên Kaggle (Save & Run All)

Mỗi lần *Save Version* chạy **một** mô hình (Qwen và Vintern cần hai phiên bản transformers khác nhau):
- Lần 1: `FAMILY = 'qwen'`
- Lần 2: `FAMILY = 'vintern'`

Cài đặt notebook giống notebook chạy thử: *GPU T4 x2*, *Internet On*, secret `GH_TOKEN`, Input = dataset ảnh.
Chạy tiếp khi bị ngắt: *Add Input → Notebook Output* của phiên bản trước; cell 2 tự chép các file `.jsonl` cũ vào và script bỏ qua ảnh đã chạy.

Cấu hình cố định cho cả hai mô hình (ghi vào báo cáo): prompt `v1`, greedy (`do_sample=False`), `repetition_penalty = 1.0`, `max_new_tokens = 768`, dtype `auto` (fp16 trên T4), Qwen `max_side = 1280`, Vintern `max_tiles = 6`.

In [ ]:
FAMILY  = 'qwen'            # 'qwen' cho lần 1, 'vintern' cho lần 2
GH_USER = 'rangha26'
BRANCH  = 'dev'
REPO    = '/tmp/V-ChartText'
OUT     = '/kaggle/working/outputs'
QWEN    = 'Qwen/Qwen3-VL-2B-Instruct'

import glob, os, unicodedata, shutil
nfc = lambda s: unicodedata.normalize('NFC', s).lower()
RAW = next(d for d in sorted(glob.glob('/kaggle/input/**/', recursive=True))
           if any('cột' in nfc(x) for x in os.listdir(d)))
print('FAMILY =', FAMILY, '| RAW =', RAW)
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# Khôi phục kết quả của phiên bản trước (nếu đã Add Input output cũ)
os.makedirs(OUT, exist_ok=True)
for f in glob.glob('/kaggle/input/**/outputs/*.jsonl', recursive=True):
    dst = os.path.join(OUT, os.path.basename(f))
    if not os.path.exists(dst):
        shutil.copy(f, dst); print('khôi phục', dst)

from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret('GH_TOKEN')
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://{token}@github.com/{GH_USER}/V-ChartText.git {REPO}
%cd {REPO}
!git log --oneline -1
if not os.path.islink('outputs'):
    !rm -rf outputs && ln -s {OUT} outputs

In [ ]:
if FAMILY == 'qwen':
    !pip install -q "transformers>=4.57.0" accelerate
else:
    !pip install -q "transformers<4.50" timm einops sentencepiece
!python -c "import transformers, torch; print('transformers', transformers.__version__, '| torch', torch.__version__)"

In [ ]:
!python src/prepare_dataset.py --raw "{RAW}"
!python eval/validate_labels.py | tail -6

In [ ]:
# extract trên GPU 0, describe trên GPU 1, chạy song song
if FAMILY == 'qwen':
    CMD = f'python src/infer_qwen.py --model {QWEN}'
else:
    CMD = 'python src/infer_vintern.py --repetition_penalty 1.0'
!CUDA_VISIBLE_DEVICES=0 {CMD} --task extract  > /kaggle/working/log_{FAMILY}_extract.txt  2>&1 & \
 CUDA_VISIBLE_DEVICES=1 {CMD} --task describe > /kaggle/working/log_{FAMILY}_describe.txt 2>&1 & wait
!tail -n 3 /kaggle/working/log_{FAMILY}_*.txt
!wc -l {OUT}/*.jsonl

In [ ]:
# Kiểm tra nhanh: số dòng, JSON hợp lệ, số output chạm max_new_tokens, thời gian trung bình
import json, re, statistics
for f in sorted(glob.glob(f'{OUT}/*.jsonl')):
    rs = [json.loads(l) for l in open(f, encoding='utf-8') if l.strip()]
    ok = cut = 0
    for r in rs:
        if (r.get('n_new_tokens') or 0) >= 760: cut += 1
        if '__extract__' in f:
            t = re.sub(r'```(?:json)?', '', r['output']); m = re.search(r'\{.*\}', t, re.S)
            try: json.loads(m.group(0)); ok += 1
            except Exception: pass
    lat = statistics.mean(r['latency_s'] for r in rs) if rs else 0
    print(f"{os.path.basename(f)}: {len(rs)} ảnh | JSON ok {ok if '__extract__' in f else '-'} | chạm giới hạn {cut} | {lat:.1f}s/ảnh")

In [ ]:
# Chấm điểm (chỉ tính ảnh đã gán nhãn) và lưu kết quả
for f in sorted(glob.glob('outputs/*__extract__*.jsonl')):
    !python eval/score_extract.py "{f}"
for f in sorted(glob.glob('outputs/*__describe__*.jsonl')):
    !python eval/check_describe.py "{f}"
!mkdir -p /kaggle/working/results && cp -r eval/results/* /kaggle/working/results/ 2>/dev/null; ls /kaggle/working/results